# Momentum on a price panel with missing quotes

In [ ]:
import numpy as np
import pandas as pd

def synthetic_prices():
    rng = np.random.default_rng(7)
    returns = rng.normal(0.0005, 0.012, size=(120, 4))
    prices = pd.DataFrame(100.0 * (1.0 + returns).cumprod(axis=0), columns=list("ABCD"))
    prices.loc[prices.index % 5 >= 3, "C"] = np.nan  # stale / missing quotes
    return prices


In [ ]:
def signal(prices):
    """10-day momentum, lagged one bar. Missing quotes are filled first."""
    filled = prices.ffill()
    momentum = filled.pct_change(10).shift(1)
    return momentum.clip(-0.2, 0.2).fillna(0.0)


In [ ]:
def backtest(prices, cost_bps=10):
    positions = signal(prices)
    asset_returns = prices.pct_change().fillna(0.0)
    gross = (positions * asset_returns).mean(axis=1)
    turnover = positions.diff().abs().sum(axis=1).fillna(0.0)
    return gross - turnover * cost_bps / 10_000

def sharpe(pnl):
    pnl_std = pnl.std()
    return float(pnl.mean() / pnl_std * np.sqrt(252)) if pnl_std > 0 else 0.0

print(round(sharpe(backtest(synthetic_prices())), 3))
